In [ ]:
import os
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

# Import your project files
from lib.networks_bp import EMCADNet
from utils.dataloader_bp import generate_bpanno

IMAGE_DIR = '/Users/sumankar/Downloads/Med2d/isic_2018/val/images'  # Change to your test image path
MASK_DIR  = '/Users/sumankar/Downloads/Med2d/isic_2018/val/masks'   # Change to your test mask path
MODEL_PATH = '/Users/sumankar/Downloads/ws_bpanno-best_isic_2018.pth'
OUTPUT_FILE = 'bpanno_visual_proof.png'
NUM_IMAGES = 4  # Number of random images to visualize

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ===========================================================================
# Helper Functions
# ===========================================================================
def load_image_and_gt(img_path, gt_path, target_size=352):
    img = np.array(Image.open(img_path).convert('RGB'))
    gt  = np.array(Image.open(gt_path).convert('L'))
    
    # Resize
    img_resized = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
    gt_resized  = cv2.resize(gt,  (target_size, target_size), interpolation=cv2.INTER_NEAREST)
    
    return img_resized, (gt_resized > 127).astype(np.float32)

def draw_weak_annotations(img, y_in, y_out, lrp_fg, lrp_bg, lrp_uncertain):
    """Draws the weak polygons on the image to show what the model is trained on."""
    vis = img.copy()
    
    # 1. Draw P_in (Certain Foreground) - Solid Green Fill
    y_in_u8 = (y_in * 255).astype(np.uint8)
    contours_in, _ = cv2.findContours(y_in_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if contours_in:
        cv2.drawContours(vis, contours_in, -1, (0, 255, 0), -1) # Green fill
        cv2.drawContours(vis, contours_in, -1, (0, 200, 0), 2)  # Dark green border

    # 2. Draw P_out (Outer Envelope) - Red Outline
    y_out_u8 = (y_out * 255).astype(np.uint8)
    contours_out, _ = cv2.findContours(y_out_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if contours_out:
        cv2.drawContours(vis, contours_out, -1, (255, 0, 0), 2) # Red outline

    # 3. Draw LRP Patches (Local Refinement) - Bright Yellow/Cyan Outlines
    lrp_mask = (lrp_fg + lrp_bg + lrp_uncertain > 0).astype(np.uint8) * 255
    contours_lrp, _ = cv2.findContours(lrp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if contours_lrp:
        cv2.drawContours(vis, contours_lrp, -1, (0, 255, 255), 2) # Cyan/Yellow outline

    # Blend with original image (50% opacity for the annotations)
    vis = cv2.addWeighted(img, 0.5, vis, 0.5, 0)
    return vis

# ===========================================================================
# Main Execution
# ===========================================================================
if __name__ == '__main__':
    print("Loading model...")
    model = EMCADNet(num_classes=1, encoder='pvt_v2_b2', pretrain=False).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    # Get random image files
    img_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png'))])
    np.random.seed(42)
    selected_files = np.random.choice(img_files, NUM_IMAGES, replace=False)

    fig, axes = plt.subplots(4, NUM_IMAGES, figsize=(20, 20))
    fig.suptitle('', 
                 fontsize=24, fontweight='bold', y=0.98)

    for col, fname in enumerate(selected_files):
        img_path = os.path.join(IMAGE_DIR, fname)
        gt_path  = os.path.join(MASK_DIR, fname.replace('.jpg', '.png'))
        
        # 1. Load Data
        img, gt = load_image_and_gt(img_path, gt_path)
        gt_np = gt.astype(np.float32)
        
        # 2. Generate Weak Annotations (This is ALL the model sees during training)
        (y_in, y_out, omega_delta, lrp_fg, lrp_bg, 
         lrp_uncertain, lrp_mask, y_c) = generate_bpanno(gt_np, K=2)
         
        # 3. Run Model Inference (Model only gets the raw image)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        # Normalize
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_tensor = (img_tensor - mean) / std
        
        with torch.no_grad():
            preds = model(img_tensor.unsqueeze(0).to(DEVICE), mode='test')
            pred_prob = torch.sigmoid(preds[-1]).squeeze().cpu().numpy()
            pred_bin = (pred_prob > 0.5).astype(np.float32)

        # --- PLOTTING ---
        
        # Row 1: Original Image
        axes[0, col].imshow(img)
        axes[0, col].set_title('Original Image', fontsize=16, fontweight='bold')
        axes[0, col].axis('off')
        
        # Row 2: Weak Annotations (The "Supervision")
        vis_weak = draw_weak_annotations(img, y_in, y_out, lrp_fg, lrp_bg, lrp_uncertain)
        axes[1, col].imshow(vis_weak)
        axes[1, col].set_title('Weak Annotations (Training Input)\nGreen=P_in, Red=P_out, Cyan=LRP', 
                               fontsize=14, fontweight='bold', color='blue')
        axes[1, col].axis('off')
        
        # Row 3: Model Prediction
        axes[2, col].imshow(pred_bin, cmap='gray')
        axes[2, col].set_title('Model Prediction (Dense Output)', 
                               fontsize=16, fontweight='bold', color='green')
        axes[2, col].axis('off')
        
        # Row 4: Ground Truth
        axes[3, col].imshow(gt, cmap='gray')
        axes[3, col].set_title('Ground Truth (Never seen in training)', 
                               fontsize=16, fontweight='bold')
        axes[3, col].axis('off')

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(OUTPUT_FILE, dpi=300, bbox_inches='tight')
    print(f"\n✅ Visualization saved to {OUTPUT_FILE}")
    print("Look at Row 2 (coarse shapes) vs Row 3 (dense mask). The model learned the dense boundary purely from the coarse shapes!")

ImportError: cannot import name 'generate_hupanno' from 'utils.dataloader_bp' (/Users/sumankar/Desktop/WS_EMCAD/utils/dataloader_bp.py)

In [8]:
import os
import cv2
import glob
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from medpy.metric.binary import hd95
import kagglehub


DATASET_PATH = os.path.abspath("/Users/sumankar/Downloads/Med2d/isic_2018/test")
MODEL_PATH_CANDIDATES = [
    "/Users/sumankar/Downloads/ws_bpanno-best_isic_2018.pth",
   
]
MODEL_PATH = next((p for p in MODEL_PATH_CANDIDATES if os.path.exists(p)), None)
if MODEL_PATH is None:
    raise FileNotFoundError(
        "No model checkpoint was found. Place a .pth file in Downloads or update MODEL_PATH_CANDIDATES."
    )

USE_TTA      = True
THRESHOLD    = 0.5
INPUT_SIZE   = 352

image_dir = os.path.join(DATASET_PATH, "images")
mask_dir_candidates = [
    os.path.join(DATASET_PATH, "masks"),
    os.path.join(DATASET_PATH, "mask"),
]

mask_dir = next((d for d in mask_dir_candidates if os.path.isdir(d)), None)
if mask_dir is None:
    raise FileNotFoundError(f"No mask directory found under {DATASET_PATH}")

image_paths = sorted(glob.glob(os.path.join(image_dir, "*")))
mask_paths = sorted(glob.glob(os.path.join(mask_dir, "*")))

print(f"Found {len(image_paths)} images and {len(mask_paths)} masks in {mask_dir}")


def normalize_name(path):
    name = os.path.basename(path)
    stem = os.path.splitext(name)[0]
    return stem.replace("_mask", "").replace("_MASK", "")


def load_mask(mask_path):
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if gt is None:
        from PIL import Image
        gt = np.array(Image.open(mask_path).convert("L"))
    return (gt > 127).astype(np.uint8)


image_lookup = {normalize_name(p): p for p in image_paths}
mask_lookup = {normalize_name(p): p for p in mask_paths}
shared_names = sorted(set(image_lookup.keys()) & set(mask_lookup.keys()))

if not shared_names:
    raise RuntimeError("No matching image/mask pairs found. Check file names.")

image_paths = [image_lookup[n] for n in shared_names]
mask_paths = [mask_lookup[n] for n in shared_names]

print(f"Using {len(shared_names)} matched pairs.")
assert len(image_paths) == len(mask_paths)

from lib.networks_bp import EMCADNet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EMCADNet(
    num_classes=1,
    kernel_sizes=[1,3,5],
    expansion_factor=2,
    dw_parallel=True,
    add=True,
    lgag_ks=3,
    activation='relu6',
    encoder='pvt_v2_b2',
    pretrain=False
)

checkpoint = torch.load(MODEL_PATH, map_location="cpu")

# Remove thop keys
checkpoint = {k: v for k, v in checkpoint.items()
              if "total_ops" not in k and "total_params" not in k}

model.load_state_dict(checkpoint, strict=False)
model = model.to(device)
model.eval()

print("Model loaded successfully!")

all_dice = []
all_iou = []
all_hd95 = []
per_image_results = []


for idx in range(len(image_paths)):

    img = cv2.imread(image_paths[idx])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    gt = load_mask(mask_paths[idx])


    resized = cv2.resize(img, (INPUT_SIZE, INPUT_SIZE))
    resized = resized.astype(np.float32) / 255.0
    resized = (resized - [0.485,0.456,0.406]) / [0.229,0.224,0.225]
    tensor = torch.tensor(resized).permute(2,0,1).unsqueeze(0).float().to(device)
    with torch.no_grad():

        p1 = model(tensor)
        if isinstance(p1, list):
            p1 = p1[-1]

        if USE_TTA:
            flipped = torch.flip(tensor, dims=[3])
            p2 = model(flipped)
            if isinstance(p2, list):
                p2 = p2[-1]
            p2 = torch.flip(p2, dims=[3])
            pred = (p1 + p2) / 2.0
        else:
            pred = p1

        pred = F.interpolate(pred, size=(h, w), mode='bilinear', align_corners=False)
        pred = torch.sigmoid(pred).squeeze().cpu().numpy()

    # -----------------------------
    # THRESHOLD
    # -----------------------------
    pred_mask = (pred >= THRESHOLD).astype(np.uint8)

    # -----------------------------
    # MORPHOLOGY (optional)
    # -----------------------------
    kernel = np.ones((3,3), np.uint8)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_OPEN, kernel)
    pred_mask = cv2.morphologyEx(pred_mask, cv2.MORPH_CLOSE, kernel)

    # -----------------------------
    # METRICS
    # -----------------------------
    intersection = (pred_mask * gt).sum()
    dice = (2*intersection + 1e-8) / (pred_mask.sum() + gt.sum() + 1e-8)

    union = pred_mask.sum() + gt.sum() - intersection
    iou = (intersection + 1e-8) / (union + 1e-8)

    try:
        if pred_mask.sum() > 0 and gt.sum() > 0:
            hd = hd95(pred_mask, gt)
        else:
            hd = 0.0
    except:
        hd = 0.0

    all_dice.append(dice)
    all_iou.append(iou)
    all_hd95.append(hd)

    per_image_results.append({
        "image": os.path.basename(image_paths[idx]),
        "dice": dice,
        "iou": iou,
        "hd95": hd
    })

    print(f"[{idx+1}/{len(image_paths)}] "
          f"{os.path.basename(image_paths[idx]):25s} "
          f"Dice={dice:.4f} IoU={iou:.4f} HD95={hd:.2f}")

# ============================================================
# FINAL STATISTICS
# ============================================================

print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print(f"Mean Dice   : {np.mean(all_dice):.4f}")
print(f"Median Dice : {np.median(all_dice):.4f}")
print(f"Std Dice    : {np.std(all_dice):.4f}")

print(f"\nMean IoU    : {np.mean(all_iou):.4f}")
print(f"Median IoU  : {np.median(all_iou):.4f}")

print(f"\nMean HD95   : {np.mean(all_hd95):.2f}")
print(f"Median HD95 : {np.median(all_hd95):.2f}")

# ============================================================
# OUTLIER DETECTION
# ============================================================

print("\nHard Cases (Dice < 0.75):")
for r in per_image_results:
    if r["dice"] < 0.75:
        print(f"{r['image']:25s} Dice={r['dice']:.4f} HD95={r['hd95']:.2f}")

Found 1000 images and 1000 masks in /Users/sumankar/Downloads/Med2d/isic_2018/test/masks
Using 1000 matched pairs.
Backbone pvt_v2_b2: 24,849,856 params
EMCAD decoder: 1,913,515 params
Model loaded successfully!
[1/1000] 000000.png                Dice=0.7917 IoU=0.6552 HD95=22.51
[2/1000] 000001.png                Dice=0.9176 IoU=0.8478 HD95=4.66
[3/1000] 000002.png                Dice=0.8497 IoU=0.7387 HD95=3.84
[4/1000] 000003.png                Dice=0.8235 IoU=0.7000 HD95=8.00
[5/1000] 000004.png                Dice=0.8550 IoU=0.7467 HD95=4.00
[6/1000] 000005.png                Dice=0.9705 IoU=0.9427 HD95=3.16
[7/1000] 000006.png                Dice=0.9539 IoU=0.9119 HD95=3.00
[8/1000] 000007.png                Dice=0.7601 IoU=0.6131 HD95=13.89
[9/1000] 000008.png                Dice=0.9582 IoU=0.9197 HD95=2.00
[10/1000] 000009.png                Dice=0.7227 IoU=0.5658 HD95=20.60
[11/1000] 000010.png                Dice=0.9311 IoU=0.8710 HD95=3.00
[12/1000] 000011.png               